# Transformers End to Ed

In [2]:
data = [
    ("i am a student", "je suis un etudiant"),
    ("how are you", "comment allez vous"),
    ("i love machine learning", "j aime apprentissage automatique"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("see you later", "a plus tard"),
    ("what is your name", "quel est votre nom"),
    ("where are you going", "ou allez vous"),
    ("i like coffee", "j aime le cafe"),
    ("welcome", "bienvenue")
]

In [4]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)

from tensorflow.keras import Model


In [5]:
# seperate input and output sentences

english_sentences = [ x [0] for x in data ] # List comprehension to extract English sentences

# add start and end tokens 

french_sentences = [ "start" + x[1] + " end" for x in data ] # List comprehension to extract French sentences and add start and end tokens

In [6]:
# Tokenization

vocab_size = 1000  # keep a maximum of 1000 words in the vocabulary

sequence_length = 20  # maximum length of the sequences


# Tokenize English sentences

source_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Tokenize French sentences

target_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

# This is where learning happpens. (Adapt the tokenizer to the data)

source_vectorization.adapt(english_sentences)

target_vectorization.adapt(french_sentences)


In [7]:
# convert Text into Numbers

encoder_inputs = source_vectorization(english_sentences)

target_tokens = target_vectorization(french_sentences)

In [8]:
# prepare decoder input and output

decoder_inputs = target_tokens[:, :-1]  # all tokens except the last one

decoder_targets = target_tokens[:, 1:]  # all tokens except the first one


In [9]:
# positional encoding

class PositionalEmbedding(tf.keras.layers.Layer):

    # constructor

    def __init__(self, sequence_length, vocab_size, embed_dim):
        # Call the parent constructor
        super().__init__()

        # Token embedding layer
        self.token_embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim)

        # Positional embedding layer
        self.position_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

        # Store the sequence length for later use
        self.sequence_length = sequence_length
    
    # call method to compute the input sequence 
    def call(self, inputs):
        
        # Get the length of the input sequence
        length = tf.shape(inputs)[-1]

        # create position numbers
        positions = tf.range(start=0, limit=length, delta=1)

        #convert word to embeddings
        embedded_tokens = self.token_embedding(inputs)

        # convert position to embeddings
        embedded_positions = self.position_embedding(positions)

        # Add both embeddings together
        return embedded_tokens + embedded_positions

In [10]:
class TransformerEncoderBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def call(self, inputs):

        # Self Attention
        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs
        )

        # Add & Norm
        proj_input = self.layernorm1(
            inputs + attention_output
        )

        # Feed Forward Network
        proj_output = self.dense_proj(
            proj_input
        )

        # Add & Norm
        return self.layernorm2(
            proj_input + proj_output
        )



In [11]:
class TransformerDecoderBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        # Masked Self-Attention
        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Cross Attention
        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Feed Forward Network
        self.ffn = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

    def call(self, inputs, encoder_outputs):

        # -----------------------
        # Masked Self Attention
        # -----------------------
        attention1_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention1_output
        )

        # -----------------------
        # Cross Attention
        # Q = Decoder
        # K,V = Encoder
        # -----------------------
        attention2_output = self.cross_attention(
            query=out1,
            value=encoder_outputs,
            key=encoder_outputs
        )

        out2 = self.layernorm2(
            out1 + attention2_output
        )

        # -----------------------
        # Feed Forward Network
        # -----------------------
        ffn_output = self.ffn(out2)

        return self.layernorm3(
            out2 + ffn_output
        )



In [12]:
embed_dim = 128
dense_dim = 512
num_heads = 4

# Encoder Input
encoder_inputs = tf.keras.Input(
    shape=(None,),
    dtype="int64",
    name="encoder_inputs"
)

x = PositionalEmbedding(
    sequence_length=sequence_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(encoder_inputs)

encoder_outputs = TransformerEncoderBlock(
    embed_dim=embed_dim,
    dense_dim=dense_dim,
    num_heads=num_heads
)(x)

# Decoder Input
decoder_inputs = tf.keras.Input(
    shape=(None,),
    dtype="int64",
    name="decoder_inputs"
)

x = PositionalEmbedding(
    sequence_length=sequence_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(decoder_inputs)

x = TransformerDecoderBlock(
    embed_dim=embed_dim,
    dense_dim=dense_dim,
    num_heads=num_heads
)(x, encoder_outputs)

# Final Vocabulary Prediction Layer
decoder_outputs = Dense(
    vocab_size,
    activation="softmax"
)(x)

# Transformer Model
transformer = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name="transformer"
)

transformer.summary()

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    130,560 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    130,560 │ decoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 128) │    396,032 │ positional_embed… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 128) │    660,096 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None,      │    129,000 │ transformer_deco… │
│                     │ 1000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,446,248 (5.52 MB)

 Trainable params: 1,446,248 (5.52 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# compile and train the model

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

encoder_input_data = source_vectorization(english_sentences)
decoder_input_data = target_tokens[:, :-1]
decoder_target_data = target_tokens[:, 1:]

transformer.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=10
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.6737 - loss: 3.9816   
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8526 - loss: 1.8232
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8526 - loss: 1.2966
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8526 - loss: 0.9736
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8737 - loss: 0.6865
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9105 - loss: 0.5447
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8789 - loss: 0.4758
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9316 - loss: 0.3697
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9368 - loss: 0.2847
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9474 - loss: 0.2174


In [17]:
# Prediction
 
test_sentence = ["i like coffee"]
 
encoder_input_test = source_vectorization(
    test_sentence
)
 
start_sentence = "start"
 
decoded_sentence = start_sentence
 
for i in range(10):
 
    tokenized_target = target_vectorization(
        [decoded_sentence]
    )
 
    predictions = transformer.predict(
        [
            encoder_input_test,
            tokenized_target
        ],
        verbose=0
    )
 
    sampled_token_index = np.argmax(
        predictions[0, i, :]
    )
 
    index_lookup = dict(
        zip(
            range(
                len(
                    target_vectorization.get_vocabulary()
                )
            ),
            target_vectorization.get_vocabulary()
        )
    )
 
    sampled_token = index_lookup[
        sampled_token_index
    ]
 
    decoded_sentence += " " + sampled_token
 
    if sampled_token == "end":
        break
 
print(decoded_sentence)

start          
